In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import os

df_raw = pd.read_csv('data/raw/YRBS_2007.csv')
initial_n = len(df_raw)

target_behavior = 'SadOrHopeless'
target_continuous = 'HowTallAreYouWithoutShoesInMeters'

df_raw['Sad_Binary'] = df_raw[target_behavior].map({1.0: 1, 2.0: 0})

missing_sad = df_raw['Sad_Binary'].isnull().sum()
missing_height = df_raw[target_continuous].isnull().sum()

df = df_raw.dropna(subset=['Sad_Binary', target_continuous]).copy()
final_n = len(df)

print(f"--- [4.2 Data Check Summary] ---")
print(f"Initial sample size: {initial_n}")
print(f"Missing/Invalid 'SadOrHopeless' records: {missing_sad}")
print(f"Missing/Invalid 'Height' records: {missing_height}")
print(f"Final sample size (n) used in analysis: {final_n}") 
print("-" * 30)

p0 = 0.30
successes = df['Sad_Binary'].sum()
phat = successes / final_n

z_stat = (phat - p0) / np.sqrt((p0 * (1 - p0)) / final_n)
p_val_z = 2 * (1 - stats.norm.cdf(abs(z_stat)))
std_error_p = np.sqrt((phat * (1 - phat)) / final_n)
ci_low_p, ci_high_p = phat - 1.96 * std_error_p, phat + 1.96 * std_error_p

print("=== One-Sample Proportion Test: Sadness ===")
print(f"Sample Proportion (p-hat): {phat:.4f}")
print(f"95% CI: [{ci_low_p:.4f}, {ci_high_p:.4f}]") 
print(f"Z-statistic: {z_stat:.4f}, P-value: {p_val_z:.6f}") 

mu0 = 1.70 
sample_mean = df[target_continuous].mean()
sample_std = df[target_continuous].std()

t_stat, p_val_t = stats.ttest_1samp(df[target_continuous], mu0)
ci_mean = stats.t.interval(0.95, final_n - 1, loc=sample_mean, scale=stats.sem(df[target_continuous]))

print("\n=== One-Sample T-Test: Height ===")
print(f"Sample Mean: {sample_mean:.4f}, Std Dev: {sample_std:.4f}")
print(f"95% CI: [{ci_mean[0]:.4f}, {ci_mean[1]:.4f}]") 
print(f"T-statistic: {t_stat:.4f}, P-value: {p_val_t:.2e}") 

data_analysis = pd.DataFrame({
    'Metric': ['Proportion_Sad', 'Mean_Height'],
    'Sample_Estimate': [phat, sample_mean],
    'Benchmark': [p0, mu0],
    'P_Value': [p_val_z, p_val_t],
    'CI_Lower': [ci_low_p, ci_mean[0]],
    'CI_Upper': [ci_high_p, ci_mean[1]]
})

data_analysis.to_csv('outputs/summary/data_analysis.csv', index=False)

--- [4.2 Data Check Summary] ---
Initial sample size: 14041
Missing/Invalid 'SadOrHopeless' records: 196
Missing/Invalid 'Height' records: 979
Final sample size (n) used in analysis: 12902
------------------------------
=== One-Sample Proportion Test: Sadness ===
Sample Proportion (p-hat): 0.2978
95% CI: [0.2899, 0.3057]
Z-statistic: -0.5494, P-value: 0.582697

=== One-Sample T-Test: Height ===
Sample Mean: 1.6941, Std Dev: 0.1015
95% CI: [1.6923, 1.6958]
T-statistic: -6.6513, P-value: 3.02e-11


The one-sample proportion test was conducted to determine if the proportion of students feeling sad or hopeless differs significantly from the benchmark of 0.30 (30%). Our analysis shows a sample proportion of 0.2978. The 95% confidence interval is [0.2899, 0.3057], which includes the benchmark value of 0.30. Since the p-value is 0.5827 (well above the 0.05 significance level), we fail to reject the null hypothesis. This suggests that there is no statistical evidence to claim the proportion of sadness in the 2007 YRBS population is significantly different from 30%.

A one-sample T-test was performed to compare the average height of students against a benchmark of 1.70 meters. The sample mean was 1.6941 meters. The 95% confidence interval is [1.6923, 1.6958], which does not include the benchmark of 1.70. With an extremely small p-value of 3.02e-11 (much less than 0.05), we reject the null hypothesis. We conclude that the average height of students in 2007 was significantly different from (specifically, slightly lower than) 1.70 meters. Even though the physical difference is small (about 0.6 cm), the large sample size makes this difference statistically highly significant.
    
Conclusion:
The analysis of the 2007 YRBS data reveals two different patterns. Regarding emotional health, the proportion of students experiencing sadness (29.78%) is statistically consistent with the 30% benchmark. However, physical measurements show that the average student height (1.694m) is statistically lower than the 1.70m benchmark. These findings satisfy the project requirements by providing both interval estimation and hypothesis testing for the selected behavior and physical variables.